# L07-01｜LoRA 实验导学


## 学习目标

完成本节后，你应当能：

1. 解释全参数微调、参数高效微调（PEFT）、LoRA 和 SFT 的关系。
2. 写出 LoRA 的低秩更新公式，说明 `rank` 与 `alpha` 的作用。
3. 读懂本实验的模型、数据、训练参数和输出目录。
4. 在 ModelArts 上选择正确的 Ascend 镜像和 Python kernel，完成环境准备。


## 1. LoRA 在训练什么

全参数微调会更新模型的每个可训练权重。以线性层为例，原始权重为 `W`，训练后变成：

`W' = W + ΔW`

LoRA 冻结 `W`，只学习一个低秩更新：

`ΔW = (alpha / r) × B × A`

其中 `A` 和 `B` 是两个小矩阵，`r` 是 `rank`。如果 `W` 的形状是 `d_out × d_in`，全参数微调需要 `d_out × d_in` 个参数，LoRA 只需要 `r × (d_in + d_out)` 个参数。`r` 越大，适配器表达能力和参数量都会增加；`alpha` 用来调整低秩分支的缩放。

本实验使用 `ms-swift` 的 SFT（Supervised Fine-Tuning，监督微调）接口。训练样本采用对话格式：用户消息提供任务，助手消息提供目标答案。训练时冻结 Qwen3-0.6B 的基础模型，只更新插入线性层的 LoRA 适配器。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('安装缺失依赖：', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
python_bin = str(Path(sys.executable).parent)
user_bin = str(Path(site.getuserbase()) / 'bin')
old_path = os.environ.get('PATH', '')
path_entries = old_path.split(os.pathsep) if old_path else []
for candidate in (python_bin, user_bin):
    if candidate not in path_entries:
        old_path = candidate + os.pathsep + old_path
        path_entries.insert(0, candidate)
os.environ['PATH'] = old_path

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    for script in candidates:
        if not script.is_file():
            continue
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
_ = torch.zeros(1, device='npu:0')

NOTEBOOK_ID = 'L07-01'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
from modelscope import snapshot_download
MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('lora_rank 控制什么？', '它是低秩分支的维度，决定适配器的参数量和表达容量。'),
    ('target_modules=all-linear 表示什么？', '它表示向模型中的线性层注入 LoRA 适配器。'),
    ('为什么要用 SFT 数据训练 LoRA？', 'SFT 用带有目标答案的对话样本，让适配器学习指定任务的输入输出关系。'),
]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
require(shutil.which('swift') is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA), 'npu_count': npu_count})
print('导学预检完成：依赖、NPU、模型和本地 JSONL 数据均已准备。')


## 2. 实验中的术语和参数

| 名称 | 含义 | 本实验中的位置 |
| --- | --- | --- |
| 基础模型（base model） | 已经预训练好的 Qwen3-0.6B 权重 | `MODEL_ID`、`MODEL_PATH` |
| 适配器（adapter） | LoRA 新增的可训练参数，训练后可单独保存 | `checkpoint-*` |
| `target_modules` | 注入 LoRA 的线性层 | 训练命令中的 `all-linear` |
| `lora_rank` | 低秩矩阵的中间维度 `r` | `LORA_RANK = 8` |
| `lora_alpha` | 低秩更新的缩放系数 | `LORA_ALPHA = 32` |
| `learning_rate` | 每次梯度更新的步长 | `LEARNING_RATE` |
| batch size | 一次前向和反向处理的样本数 | `PER_DEVICE_BATCH_SIZE` |
| gradient accumulation | 累积多次梯度后再更新参数 | `GRADIENT_ACCUMULATION_STEPS` |
| `max_length` | 输入序列的最大 token 数 | `MAX_LENGTH` |
| step | 一次参数更新 | 日志中的 `global_step` |
| epoch | 完整遍历一次训练集 | `NUM_TRAIN_EPOCHS` |
| checkpoint | 某个训练时刻保存的适配器和状态 | `checkpoint-*` |

`loss` 是当前训练批次的目标函数值。SFT 中，模型根据上下文预测目标 token，交叉熵通常用作训练 loss。训练日志里的 loss 是优化过程的观测值；L07-03 会把它画成曲线。

### 参数之间的关系

一次参数更新处理的样本量可以近似写成：

`effective batch size = per-device batch size × gradient accumulation steps × data parallel size`

本实验是单卡运行，`PER_DEVICE_BATCH_SIZE=1`、`GRADIENT_ACCUMULATION_STEPS=1`，所以一次 step 处理 1 条训练记录。调参时先固定模型和数据，再一次修改一个参数。

### 一个 rank 计算例子

假设某个线性层是 `4096 × 4096`，全参数微调需要约 16.8M 个参数。`rank=8` 的 LoRA 分支只需要 `8 × (4096 + 4096) = 65,536` 个参数。实际模型中不同线性层的维度不同，但计算方法相同。

本节代码会准备依赖、NPU、模型和 JSONL 数据。运行结束后，记下输出中的 `model_path` 和 `train_data`，后面两节会用同样的目录结构。

## 3. 本实验使用的训练样本

数据文件是 JSONL，每一行包含一个 `messages` 列表：

```json
{
  "messages": [
    {"role": "system", "content": "你是一个简洁、准确的课程实验助手。"},
    {"role": "user", "content": "请解释 LoRA。"},
    {"role": "assistant", "content": "LoRA 只训练低秩适配器参数。"}
  ]
}
```

`system` 定义角色，`user` 提出问题，`assistant` 提供监督答案。Qwen3 的思考格式在本实验中用空的 `<think>` 段落和 `/no_think` 控制，避免把思考过程作为训练目标。

训练数据少时，LoRA 适合用来快速观察参数更新和训练流程；正式任务仍需要更有代表性的数据集和独立评估。

## 4. ModelArts 上的运行路径

1. 在 ModelArts 创建 Notebook，选择本实验 README 中列出的 Ascend 镜像和规格。
2. 打开 JupyterLab，在右上角选择 `Python (PyTorch-2.7.1)` kernel。
3. 把三个 notebook 放入工作目录，按 `L07-01 → L07-02 → L07-03` 的顺序运行。
4. 本节运行结束后，确认输出中能看到 NPU 数量、模型目录和训练数据路径。
5. 进入 L07-02，观察一条 LoRA 训练命令是怎样由这些参数组成的。

## 5. 课前练习

1. 在公式 `ΔW = (alpha / r) × B × A` 中，`r` 变大后，哪一部分的参数量会增加？
2. 为什么 LoRA 可以冻结基础模型，只训练适配器？
3. `lora_alpha` 和 `learning_rate` 都会影响训练过程，它们分别作用在哪一层？
4. 如果把 `target_modules` 从 `all-linear` 改为只训练 `q_proj` 和 `v_proj`，可训练参数量会怎样变化？

带着这些问题进入 L07-02。


## 本节小结

本节建立了 LoRA 实验的共同语言：基础模型提供原始能力，LoRA 通过低秩矩阵学习任务相关的权重增量，SFT 用带答案的对话样本训练这些增量。下一节先用 3 个 step 检查这条链路，再在 L07-03 中完成 50 个 step 的训练并分析 loss。
